## AAM / Machine Learning - Atividade 02 - Competição Porto Seguro

**Equipe**:

*Ciro Mora* - RA: 111310

*José Diniz* - RA: 183134

*Renan Renger* - RA: 183148

*Roberto Rodrigues* -  RA: 060235

**Projeto**

Predição de acionamento ou não de sinistro de seguro baseado em dataset liberado pela competição.

**Observação**

Boa parte dos comentário do notebook são proveniente de um notebook base fornecido em sala com a descrição da atividade serão mantidos por comodidade.


## Pré-processamento é provavelmente a parte mais importante de ciência dos dados

Ter dados representativos sem atributos faltantes é provavelmente o pote de ouro em ciência dos dados. É muito incomum que os dados do mundo real não apresentem anomalias seja da própria natureza ou sejam anomalias introduzidas no processo de medição e registro da observação (amostra).

Esse notebook é voltado para como tratar dados mais complexos e transformar todas as informações em números que façam sentido para que o modelo seja capaz de traçar a relação entre atributos e classes. A seguir é oferecida uma pequena parcela de um conjunto de dados da empresa Porto Seguro, no qual uma competição foi aberta e os competidores foram desafiados a criar um modelo para prever se uma apólice teria um sinistro registrado ou não, indicando o uso do serviço.

Algumas características sobre o nome das features:
- O nome dos atributos indica o grupo ao qual pertence (ind, reg, car);
- Os prefixos bin e cat indicam atributos binários e categóricos, respectivamente;
- Atributos sem os prefixos citados podem ser ordinais ou contínuos;
- Atributos com -1 indicam dado faltante (missing); e
- A coluna 'target' indica se houve sinistro para apólice ou não.

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
import plotly.offline as py
py.init_notebook_mode(connected=True)
import plotly.graph_objs as go

import warnings
warnings.filterwarnings('ignore')

from numba import jit

In [2]:
@jit
def eval_gini(y_true, y_prob):
    """
    Original author CPMP : https://www.kaggle.com/cpmpml
    In kernel : https://www.kaggle.com/cpmpml/extremely-fast-gini-computation
    """
    y_true = np.asarray(y_true)
    y_true = y_true[np.argsort(y_prob)]
    ntrue = 0
    gini = 0
    delta = 0
    n = len(y_true)
    for i in range(n-1, -1, -1):
        y_i = y_true[i]
        ntrue += y_i
        gini += y_i * delta
        delta += 1 - y_i
    gini = 1 - 2 * gini / (ntrue * (n - ntrue))
    return gini

In [3]:
dfTreino = pd.read_csv('../input/train.csv')
dfTeste = pd.read_csv('../input/test.csv')

In [4]:
# Podemos usar a coluna "id" como o índice dos Datasets, sem nenhum prejuizo
dfTreino.set_index('id', inplace=True)
dfTeste.set_index('id', inplace=True)

In [5]:
headNumber = 5
print(f'Dataset de treino - Primeiras {headNumber} linhas')
display(dfTreino.head(headNumber))

print(f'Dataset de teste - Primeiras {headNumber} linhas')
display(dfTeste.head(headNumber))

Dataset de treino - Primeiras 5 linhas


,target,ps_ind_01,ps_ind_02_cat,ps_ind_03,ps_ind_04_cat,ps_ind_05_cat,ps_ind_06_bin,ps_ind_07_bin,ps_ind_08_bin,ps_ind_09_bin,ps_ind_10_bin,ps_ind_11_bin,ps_ind_12_bin,ps_ind_13_bin,ps_ind_14,ps_ind_15,ps_ind_16_bin,ps_ind_17_bin,ps_ind_18_bin,ps_reg_01,ps_reg_02,ps_reg_03,ps_car_01_cat,ps_car_02_cat,ps_car_03_cat,ps_car_04_cat,ps_car_05_cat,ps_car_06_cat,ps_car_07_cat,ps_car_08_cat,ps_car_09_cat,ps_car_10_cat,ps_car_11_cat,ps_car_11,ps_car_12,ps_car_13,ps_car_14,ps_car_15,ps_calc_01,ps_calc_02,ps_calc_03,ps_calc_04,ps_calc_05,ps_calc_06,ps_calc_07,ps_calc_08,ps_calc_09,ps_calc_10,ps_calc_11,ps_calc_12,ps_calc_13,ps_calc_14,ps_calc_15_bin,ps_calc_16_bin,ps_calc_17_bin,ps_calc_18_bin,ps_calc_19_bin,ps_calc_20_bin
id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
7,0,2,2,5,1,0,0,1,0,0,0,0,0,0,0,11,0,1,0,0.7,0.2,0.718070,10,1,-1,0,1,4,1,0,0,1,12,2,0.400000,0.883679,0.370810,3.605551,0.6,0.5,0.2,3,1,10,1,10,1,5,9,1,5,8,0,1,1,0,0,1
9,0,1,1,7,0,0,0,0,1,0,0,0,0,0,0,3,0,0,1,0.8,0.4,0.766078,11,1,-1,0,-1,11,1,1,2,1,19,3,0.316228,0.618817,0.388716,2.449490,0.3,0.1,0.3,2,1,9,5,8,1,7,3,1,1,9,0,1,1,0,1,0
13,0,5,4,9,1,0,0,0,1,0,0,0,0,0,0,12,1,0,0,0.0,0.0,-1.000000,7,1,-1,0,-1,14,1,1,2,1,60,1,0.316228,0.641586,0.347275,3.316625,0.5,0.7,0.1,2,2,9,1,8,2,7,4,2,7,7,0,1,1,0,1,0
16,0,0,1,2,0,0,1,0,0,0,0,0,0,0,0,8,1,0,0,0.9,0.2,0.580948,7,1,0,0,1,11,1,1,3,1,104,1,0.374166,0.542949,0.294958,2.000000,0.6,0.9,0.1,2,4,7,1,8,4,2,2,2,4,9,0,0,0,0,0,0
17,0,0,2,0,1,0,1,0,0,0,0,0,0,0,0,9,1,0,0,0.7,0.6,0.840759,11,1,-1,0,-1,14,1,1,2,1,82,3,0.316070,0.565832,0.365103,2.000000,0.4,0.6,0.0,2,2,6,3,10,2,12,3,1,1,3,0,0,0,1,1,0


Dataset de teste - Primeiras 5 linhas


,ps_ind_01,ps_ind_02_cat,ps_ind_03,ps_ind_04_cat,ps_ind_05_cat,ps_ind_06_bin,ps_ind_07_bin,ps_ind_08_bin,ps_ind_09_bin,ps_ind_10_bin,ps_ind_11_bin,ps_ind_12_bin,ps_ind_13_bin,ps_ind_14,ps_ind_15,ps_ind_16_bin,ps_ind_17_bin,ps_ind_18_bin,ps_reg_01,ps_reg_02,ps_reg_03,ps_car_01_cat,ps_car_02_cat,ps_car_03_cat,ps_car_04_cat,ps_car_05_cat,ps_car_06_cat,ps_car_07_cat,ps_car_08_cat,ps_car_09_cat,ps_car_10_cat,ps_car_11_cat,ps_car_11,ps_car_12,ps_car_13,ps_car_14,ps_car_15,ps_calc_01,ps_calc_02,ps_calc_03,ps_calc_04,ps_calc_05,ps_calc_06,ps_calc_07,ps_calc_08,ps_calc_09,ps_calc_10,ps_calc_11,ps_calc_12,ps_calc_13,ps_calc_14,ps_calc_15_bin,ps_calc_16_bin,ps_calc_17_bin,ps_calc_18_bin,ps_calc_19_bin,ps_calc_20_bin
id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0,0,1,8,1,0,0,1,0,0,0,0,0,0,0,12,1,0,0,0.5,0.3,0.610328,7,1,-1,0,-1,1,1,1,2,1,65,1,0.316228,0.669556,0.352136,3.464102,0.1,0.8,0.6,1,1,6,3,6,2,9,1,1,1,12,0,1,1,0,0,1
1,4,2,5,1,0,0,0,0,1,0,0,0,0,0,5,1,0,0,0.9,0.5,0.771362,4,1,-1,0,0,11,1,1,0,1,103,1,0.316228,0.606320,0.358329,2.828427,0.4,0.5,0.4,3,3,8,4,10,2,7,2,0,3,10,0,0,1,1,0,1
2,5,1,3,0,0,0,0,0,1,0,0,0,0,0,10,0,0,0,0.4,0.0,0.916174,11,1,-1,0,-1,14,1,1,2,1,29,3,0.400000,0.896239,0.398497,3.316625,0.6,0.6,0.6,2,3,7,4,6,3,12,4,0,2,4,0,0,0,0,0,0
3,0,1,6,0,0,1,0,0,0,0,0,0,0,0,4,1,0,0,0.1,0.2,-1.000000,7,1,-1,0,-1,1,1,1,2,1,40,2,0.374166,0.652110,0.381445,2.449490,0.1,0.5,0.5,2,1,7,3,12,1,13,5,1,0,5,1,0,1,0,0,0
4,5,1,7,0,0,0,0,0,1,0,0,0,0,0,4,1,0,0,0.9,0.4,0.817771,11,1,-1,0,-1,11,1,1,2,1,101,3,0.374166,0.812914,0.385097,3.316625,0.9,0.6,0.8,3,4,7,1,10,4,12,4,0,0,4,0,1,1,0,0,1


In [6]:
print('Dataset de treino - Estatistica descritiva')
display(dfTreino.describe())

print('Dataset de teste - Estatistica descritiva')
display(dfTeste.describe())

Dataset de treino - Estatistica descritiva


,target,ps_ind_01,ps_ind_02_cat,ps_ind_03,ps_ind_04_cat,ps_ind_05_cat,ps_ind_06_bin,ps_ind_07_bin,ps_ind_08_bin,ps_ind_09_bin,ps_ind_10_bin,ps_ind_11_bin,ps_ind_12_bin,ps_ind_13_bin,ps_ind_14,ps_ind_15,ps_ind_16_bin,ps_ind_17_bin,ps_ind_18_bin,ps_reg_01,ps_reg_02,ps_reg_03,ps_car_01_cat,ps_car_02_cat,ps_car_03_cat,ps_car_04_cat,ps_car_05_cat,ps_car_06_cat,ps_car_07_cat,ps_car_08_cat,ps_car_09_cat,ps_car_10_cat,ps_car_11_cat,ps_car_11,ps_car_12,ps_car_13,ps_car_14,ps_car_15,ps_calc_01,ps_calc_02,ps_calc_03,ps_calc_04,ps_calc_05,ps_calc_06,ps_calc_07,ps_calc_08,ps_calc_09,ps_calc_10,ps_calc_11,ps_calc_12,ps_calc_13,ps_calc_14,ps_calc_15_bin,ps_calc_16_bin,ps_calc_17_bin,ps_calc_18_bin,ps_calc_19_bin,ps_calc_20_bin
count,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000,595212.000000
mean,0.036448,1.900378,1.358943,4.423318,0.416794,0.405188,0.393742,0.257033,0.163921,0.185304,0.000373,0.001692,0.009439,0.000948,0.012451,7.299922,0.660823,0.121081,0.153446,0.610991,0.439184,0.551102,8.295933,0.829931,-0.504899,0.725192,-0.157732,6.555340,0.910027,0.832080,1.328890,0.992136,62.215674,2.346072,0.379945,0.813265,0.276256,3.065899,0.449756,0.449589,0.449849,2.372081,1.885886,7.689445,3.005823,9.225904,2.339034,8.433590,5.441382,1.441918,2.872288,7.539026,0.122427,0.627840,0.554182,0.287182,0.349024,0.153318
std,0.187401,1.983789,0.664594,2.699902,0.493311,1.350642,0.488579,0.436998,0.370205,0.388544,0.019309,0.041097,0.096693,0.030768,0.127545,3.546042,0.473430,0.326222,0.360417,0.287643,0.404264,0.793506,2.508270,0.375716,0.788654,2.153463,0.844417,5.501445,0.347106,0.373796,0.978747,0.091619,33.012455,0.832548,0.058327,0.224588,0.357154,0.731366,0.287198,0.286893,0.287153,1.117219,1.134927,1.334312,1.414564,1.459672,1.246949,2.904597,2.332871,1.202963,1.694887,2.746652,0.327779,0.483381,0.497056,0.452447,0.476662,0.360295
min,0.000000,0.000000,-1.000000,0.000000,-1.000000,-1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-1.000000,-1.000000,-1.000000,-1.000000,0.000000,-1.000000,0.000000,-1.000000,0.000000,-1.000000,0.000000,1.000000,-1.000000,-1.000000,0.250619,-1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,1.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,5.000000,0.000000,0.000000,0.000000,0.400000,0.200000,0.525000,7.000000,1.000000,-1.000000,0.000000,-1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,32.000000,2.000000,0.316228,0.670867,0.333167,2.828427,0.200000,0.200000,0.200000,2.000000,1.000000,7.000000,2.000000,8.000000,1.000000,6.000000,4.000000,1.000000,2.000000,6.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,1.000000,1.000000,4.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,7.000000,1.000000,0.000000,0.000000,0.700000,0.300000,0.720677,7.000000,1.000000,-1.000000,0.000000,0.000000,7.000000,1.000000,1.000000,2.000000,1.000000,65.000000,3.000000,0.374166,0.765811,0.368

Dataset de teste - Estatistica descritiva


,ps_ind_01,ps_ind_02_cat,ps_ind_03,ps_ind_04_cat,ps_ind_05_cat,ps_ind_06_bin,ps_ind_07_bin,ps_ind_08_bin,ps_ind_09_bin,ps_ind_10_bin,ps_ind_11_bin,ps_ind_12_bin,ps_ind_13_bin,ps_ind_14,ps_ind_15,ps_ind_16_bin,ps_ind_17_bin,ps_ind_18_bin,ps_reg_01,ps_reg_02,ps_reg_03,ps_car_01_cat,ps_car_02_cat,ps_car_03_cat,ps_car_04_cat,ps_car_05_cat,ps_car_06_cat,ps_car_07_cat,ps_car_08_cat,ps_car_09_cat,ps_car_10_cat,ps_car_11_cat,ps_car_11,ps_car_12,ps_car_13,ps_car_14,ps_car_15,ps_calc_01,ps_calc_02,ps_calc_03,ps_calc_04,ps_calc_05,ps_calc_06,ps_calc_07,ps_calc_08,ps_calc_09,ps_calc_10,ps_calc_11,ps_calc_12,ps_calc_13,ps_calc_14,ps_calc_15_bin,ps_calc_16_bin,ps_calc_17_bin,ps_calc_18_bin,ps_calc_19_bin,ps_calc_20_bin
count,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.00000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000,892816.000000
mean,1.902371,1.358613,4.413734,0.417361,0.408132,0.393246,0.257191,0.163659,0.185905,0.000373,0.001595,0.009376,0.001039,0.012383,7.296862,0.660591,0.120399,0.154973,0.611057,0.439868,0.551740,8.292038,0.82994,-0.504895,0.725798,-0.158449,6.564297,0.910144,0.832311,1.327911,0.992134,62.284293,2.346606,0.379957,0.813637,0.276431,3.068325,0.449632,0.450453,0.450054,2.371390,1.885328,7.687805,3.009537,9.225854,2.338538,8.443167,5.438478,1.440265,2.875013,7.540367,0.123720,0.627756,0.554660,0.287796,0.349344,0.152428
std,1.986503,0.663002,2.700149,0.493453,1.355068,0.488471,0.437086,0.369966,0.389030,0.019309,0.039905,0.096375,0.032223,0.127251,3.541948,0.473509,0.325428,0.361879,0.287699,0.404678,0.794023,2.508331,0.37570,0.788753,2.154886,0.844566,5.500485,0.347283,0.373590,0.978741,0.091530,33.003714,0.831666,0.058385,0.224779,0.356835,0.729005,0.287213,0.287373,0.287254,1.116953,1.136763,1.333521,1.415155,1.460560,1.248601,2.906361,2.330081,1.200620,1.694072,2.745882,0.329262,0.483403,0.497004,0.452736,0.476763,0.359435
min,0.000000,-1.000000,0.000000,-1.000000,-1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-1.000000,-1.000000,-1.00000,-1.000000,0.000000,-1.000000,0.000000,-1.000000,0.000000,-1.000000,0.000000,1.000000,-1.000000,0.141421,0.275778,-1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,1.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,5.000000,0.000000,0.000000,0.000000,0.400000,0.200000,0.524404,7.000000,1.00000,-1.000000,0.000000,-1.000000,1.000000,1.000000,1.000000,0.000000,1.000000,32.000000,2.000000,0.316228,0.671234,0.333916,2.828427,0.200000,0.200000,0.200000,2.000000,1.000000,7.000000,2.000000,8.000000,1.000000,6.000000,4.000000,1.000000,2.000000,6.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,1.000000,4.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,7.000000,1.000000,0.000000,0.000000,0.700000,0.300000,0.721110,7.000000,1.00000,-1.000000,0.000000,0.000000,7.000000,1.000000,1.000000,2.000000,1.000000,65.000000,3.000000,0.374166,0.766081,0.368782,3.316625,0.400000,0.500000,0.400000,2.000000,2.000000,8.000000,3.0000

In [7]:
print('Dataset de treino - Sumário das Features')
print(dfTreino.info())
print('---')
print('Dataset de treino - Sumário das Features')
print(dfTeste.info())

Dataset de treino - Sumário das Features
<class 'pandas.core.frame.DataFrame'>
Int64Index: 595212 entries, 7 to 1488027
Data columns (total 58 columns):
target            595212 non-null int64
ps_ind_01         595212 non-null int64
ps_ind_02_cat     595212 non-null int64
ps_ind_03         595212 non-null int64
ps_ind_04_cat     595212 non-null int64
ps_ind_05_cat     595212 non-null int64
ps_ind_06_bin     595212 non-null int64
ps_ind_07_bin     595212 non-null int64
ps_ind_08_bin     595212 non-null int64
ps_ind_09_bin     595212 non-null int64
ps_ind_10_bin     595212 non-null int64
ps_ind_11_bin     595212 non-null int64
ps_ind_12_bin     595212 non-null int64
ps_ind_13_bin     595212 non-null int64
ps_ind_14         595212 non-null int64
ps_ind_15         595212 non-null int64
ps_ind_16_bin     595212 non-null int64
ps_ind_17_bin     595212 non-null int64
ps_ind_18_bin     595212 non-null int64
ps_reg_01         595212 non-null float64
ps_reg_02         595212 non-null float64
ps_

In [8]:
print(f'Dataset de treino tem {dfTreino.shape[0]} linhas por {dfTreino.shape[1]} colunas ({dfTreino.shape[0] * dfTreino.shape[1]} celulas)')
print(f'Dataset de teste tem {dfTeste.shape[0]} linhas por {dfTeste.shape[1]} colunas ({dfTeste.shape[0] * dfTeste.shape[1]} celulas)')

Dataset de treino tem 595212 linhas por 58 colunas (34522296 celulas)
Dataset de teste tem 892816 linhas por 57 colunas (50890512 celulas)


In [9]:
nonUsed, used = dfTreino.groupby('target').size()
print(f'Das {dfTreino.shape[0]} entradas no dataset, {nonUsed} foram de casos onde não foi acionado o seguro e {used} foram caso onde houve acionamento')
print(f'Temos assim {round((used/nonUsed) * 100,6)}% de ocorrencias em que o resultado (1 ou "houve acionamento") desejamos prever')

Das 595212 entradas no dataset, 573518 foram de casos onde não foi acionado o seguro e 21694 foram caso onde houve acionamento
Temos assim 3.782619% de ocorrencias em que o resultado (1 ou "houve acionamento") desejamos prever


## Observação
Com base nas analises acima, podemos perceber algumas coisas:

1 -  Os valores estão desnormalizados, variando tanto de tipo (numéricos discretos e continuos, categoricos e binários);

2 - Faltando dados em ambos os datasets (marcados como -1 nos datasets);

3 - **Temos um enorme desbalanço no que tange a ocorrencias cujo valor é desejado ("houve acionamento") vs ocorrencias sem acionamento**

---

In [10]:
print(f'Antes - Treino tem {dfTreino.shape[0]} linhas por {dfTreino.shape[1]} colunas ({dfTreino.shape[0] * dfTreino.shape[1]} celulas)')
dfTreino.drop_duplicates()
print(f'Depois - Treino tem {dfTreino.shape[0]} linhas por {dfTreino.shape[1]} colunas ({dfTreino.shape[0] * dfTreino.shape[1]} celulas)')
print('---')
print(f'Antes - Teste tem {dfTeste.shape[0]} linhas por {dfTeste.shape[1]} colunas ({dfTeste.shape[0] * dfTeste.shape[1]} celulas)')
dfTeste.drop_duplicates()
print(f'Depois - Teste tem {dfTeste.shape[0]} linhas por {dfTeste.shape[1]} colunas ({dfTeste.shape[0] * dfTeste.shape[1]} celulas)')

Antes - Treino tem 595212 linhas por 58 colunas (34522296 celulas)
Depois - Treino tem 595212 linhas por 58 colunas (34522296 celulas)
---
Antes - Teste tem 892816 linhas por 57 colunas (50890512 celulas)
Depois - Teste tem 892816 linhas por 57 colunas (50890512 celulas)


Ao trabalhar com as colunas (atributos), é interessante ter uma organização de que tipo de dado determinado atributo é, e para quais propósitos determinado atributo pode ser usado. Nesse sentido, seguindo o trabalho de https://www.kaggle.com/bertcarremans/data-preparation-exploration, vamos criar metadados para esse conjunto.

In [11]:
def generateMetadata(dfInput):
    data = []
    for f in dfInput.columns:
        # definindo o uso (entre rótulo, id e atributos)
        if f == 'target':
            role = 'target' # rótulo
        elif f == 'id':
            role = 'id'
        else:
            role = 'input' # atributos

        # definindo o tipo do dado
        if 'bin' in f or f == 'target':
            level = 'binary'
        elif 'cat' in f or f == 'id':
            level = 'nominal'
        elif dfInput[f].dtype == float or dfInput[f].dtype == np.float64:
            level = 'interval'
        elif dfInput[f].dtype == int or dfInput[f].dtype == np.int64:
            level = 'ordinal'
            
        # mantem keep como verdadeiro pra tudo, exceto id
        keep = True
        if f == 'id':
            keep = False

        # cria o tipo de dado
        dtype = dfInput[f].dtype

        # cria dicionário de metadados
        f_dict = {
            'varname': f,
            'role': role,
            'level': level,
            'keep': keep,
            'dtype': dtype
        }
        data.append(f_dict)

    meta = pd.DataFrame(data, columns=['varname', 'role', 'level', 'keep', 'dtype'])
    meta.set_index('varname', inplace=True)
    
    return meta

Para visualizar o atributo e todos seus metadados, basta mostrar a variável meta:

In [12]:
meta_train = generateMetadata(dfTreino)
meta_test = generateMetadata(dfTeste)

In [13]:
display(meta_train)
display(meta_test)

,role,level,keep,dtype
varname,,,,
target,target,binary,True,int64
ps_ind_01,input,ordinal,True,int64
ps_ind_02_cat,input,nominal,True,int64
ps_ind_03,input,ordinal,True,int64
ps_ind_04_cat,input,nominal,True,int64
ps_ind_05_cat,input,nominal,True,int64
ps_ind_06_bin,input,binary,True,int64
ps_ind_07_bin,input,binary,True,int64
ps_ind_08_bin,input,binary,True,int64


,role,level,keep,dtype
varname,,,,
ps_ind_01,input,ordinal,True,int64
ps_ind_02_cat,input,nominal,True,int64
ps_ind_03,input,ordinal,True,int64
ps_ind_04_cat,input,nominal,True,int64
ps_ind_05_cat,input,nominal,True,int64
ps_ind_06_bin,input,binary,True,int64
ps_ind_07_bin,input,binary,True,int64
ps_ind_08_bin,input,binary,True,int64
ps_ind_09_bin,input,binary,True,int64


Com essa estrutura de metadados, fica fácil consultar quais colunas quer se manter e que são nominais, por exemplo:

In [14]:
print('Metadados categoricos da base de treino')
print(meta_train[(meta_train.level == 'nominal') & (meta_train.keep)].index)
print('---')
print('Metadados categoricos da base de teste')
print(meta_test[(meta_test.level == 'nominal') & (meta_test.keep)].index)

Metadados categoricos da base de treino
Index(['ps_ind_02_cat', 'ps_ind_04_cat', 'ps_ind_05_cat', 'ps_car_01_cat',
       'ps_car_02_cat', 'ps_car_03_cat', 'ps_car_04_cat', 'ps_car_05_cat',
       'ps_car_06_cat', 'ps_car_07_cat', 'ps_car_08_cat', 'ps_car_09_cat',
       'ps_car_10_cat', 'ps_car_11_cat'],
      dtype='object', name='varname')
---
Metadados categoricos da base de teste
Index(['ps_ind_02_cat', 'ps_ind_04_cat', 'ps_ind_05_cat', 'ps_car_01_cat',
       'ps_car_02_cat', 'ps_car_03_cat', 'ps_car_04_cat', 'ps_car_05_cat',
       'ps_car_06_cat', 'ps_car_07_cat', 'ps_car_08_cat', 'ps_car_09_cat',
       'ps_car_10_cat', 'ps_car_11_cat'],
      dtype='object', name='varname')


Da mesma forma, seria possível contar os atributos por tipo de uso e dado:

In [15]:
print('Tipos e quantidade de features do dataset de treino')
display(pd.DataFrame({'count' : meta_train.groupby(['role', 'level'])['role'].size()}).reset_index())

print('Tipos e quantidade de features do dataset de teste')
display(pd.DataFrame({'count' : meta_test.groupby(['role', 'level'])['role'].size()}).reset_index())

Tipos e quantidade de features do dataset de treino


,role,level,count
0,input,binary,17
1,input,interval,10
2,input,nominal,14
3,input,ordinal,16
4,target,binary,1


Tipos e quantidade de features do dataset de teste


,role,level,count
0,input,binary,17
1,input,interval,10
2,input,nominal,14
3,input,ordinal,16


## Valores faltantes

Conforme já mencionado, os valores faltantes são indicados por -1, então é importante saber quais colunas têm valores faltantes e em qual proporção.

In [16]:
def getMissingAttributes(dfInput):
    atributos_missing = []
    return_missing = []

    for f in dfInput.columns:
        missings = dfInput[dfInput[f] == -1][f].count()
        if missings > 0:
            atributos_missing.append(f)
            missings_perc = missings/dfInput.shape[0]
            
            return_missing.append([f, missings, missings_perc])

            print('Atributo {} tem {} amostras ({:.2%}) com valores faltantes'.format(f, missings, missings_perc))
            

    print('No total, há {} atributos com valores faltantes'.format(len(atributos_missing)))
    
    return pd.DataFrame(return_missing).rename(index=str, columns={0: "column_name", 1: "column_nulls", 2: "column_percentage"})

In [17]:
missing_Train = getMissingAttributes(dfTreino)
display(missing_Train)

Atributo ps_ind_02_cat tem 216 amostras (0.04%) com valores faltantes
Atributo ps_ind_04_cat tem 83 amostras (0.01%) com valores faltantes
Atributo ps_ind_05_cat tem 5809 amostras (0.98%) com valores faltantes
Atributo ps_reg_03 tem 107772 amostras (18.11%) com valores faltantes
Atributo ps_car_01_cat tem 107 amostras (0.02%) com valores faltantes
Atributo ps_car_02_cat tem 5 amostras (0.00%) com valores faltantes
Atributo ps_car_03_cat tem 411231 amostras (69.09%) com valores faltantes
Atributo ps_car_05_cat tem 266551 amostras (44.78%) com valores faltantes
Atributo ps_car_07_cat tem 11489 amostras (1.93%) com valores faltantes
Atributo ps_car_09_cat tem 569 amostras (0.10%) com valores faltantes
Atributo ps_car_11 tem 5 amostras (0.00%) com valores faltantes
Atributo ps_car_12 tem 1 amostras (0.00%) com valores faltantes
Atributo ps_car_14 tem 42620 amostras (7.16%) com valores faltantes
No total, há 13 atributos com valores faltantes


,column_name,column_nulls,column_percentage
0,ps_ind_02_cat,216,0.000363
1,ps_ind_04_cat,83,0.000139
2,ps_ind_05_cat,5809,0.009760
3,ps_reg_03,107772,0.181065
4,ps_car_01_cat,107,0.000180
5,ps_car_02_cat,5,0.000008
6,ps_car_03_cat,411231,0.690898
7,ps_car_05_cat,266551,0.447825
8,ps_car_07_cat,11489,0.019302
9,ps_car_09_cat,569,0.000956


In [18]:
missing_Test = getMissingAttributes(dfTeste)
display(missing_Test)

Atributo ps_ind_02_cat tem 307 amostras (0.03%) com valores faltantes
Atributo ps_ind_04_cat tem 145 amostras (0.02%) com valores faltantes
Atributo ps_ind_05_cat tem 8710 amostras (0.98%) com valores faltantes
Atributo ps_reg_03 tem 161684 amostras (18.11%) com valores faltantes
Atributo ps_car_01_cat tem 160 amostras (0.02%) com valores faltantes
Atributo ps_car_02_cat tem 5 amostras (0.00%) com valores faltantes
Atributo ps_car_03_cat tem 616911 amostras (69.10%) com valores faltantes
Atributo ps_car_05_cat tem 400359 amostras (44.84%) com valores faltantes
Atributo ps_car_07_cat tem 17331 amostras (1.94%) com valores faltantes
Atributo ps_car_09_cat tem 877 amostras (0.10%) com valores faltantes
Atributo ps_car_11 tem 1 amostras (0.00%) com valores faltantes
Atributo ps_car_14 tem 63805 amostras (7.15%) com valores faltantes
No total, há 12 atributos com valores faltantes


,column_name,column_nulls,column_percentage
0,ps_ind_02_cat,307,0.000344
1,ps_ind_04_cat,145,0.000162
2,ps_ind_05_cat,8710,0.009756
3,ps_reg_03,161684,0.181094
4,ps_car_01_cat,160,0.000179
5,ps_car_02_cat,5,0.000006
6,ps_car_03_cat,616911,0.690972
7,ps_car_05_cat,400359,0.448423
8,ps_car_07_cat,17331,0.019412
9,ps_car_09_cat,877,0.000982


Duas estratégias podem ser optadas aqui: simplesmente remover o atributo ou tentar preenchê-lo de forma sintética. Preencher de forma sintética pode gerar uma falsa distribuição quando o número de atributos faltantes é muito alto. Quando este for o caso, é sempre seguro optar por remover o atributo inteiro. Também é importante lembrar que a estratégia de preenchimento deve ser coerente com o tipo de dado, por exemplo: **dados ordinais não devem ser preenchidos com média, nem dados contínuos com moda.**

In [19]:
# limiar de remoção - 42.5% de nulos
remove_threshold = 0.425

In [20]:
columns_to_remove = np.array(missing_Train.column_name[(missing_Train.column_percentage >= remove_threshold)])

In [21]:
# removendo as colunas que tem muitos valores faltantes
dfTreino = dfTreino.drop(columns_to_remove, axis=1)
dfTeste = dfTeste.drop(columns_to_remove, axis=1)

# atualiza os metadados para ter como referência
meta_train.loc[(columns_to_remove),'keep'] = False  
meta_test.loc[(columns_to_remove),'keep'] = False

# remove do frame de colunas com falta de dados as colunas que foram dropadas
missing_Train.drop(missing_Train[(np.isin(missing_Train.column_name, columns_to_remove))].index)

,column_name,column_nulls,column_percentage
0,ps_ind_02_cat,216,0.000363
1,ps_ind_04_cat,83,0.000139
2,ps_ind_05_cat,5809,0.009760
3,ps_reg_03,107772,0.181065
4,ps_car_01_cat,107,0.000180
5,ps_car_02_cat,5,0.000008
8,ps_car_07_cat,11489,0.019302
9,ps_car_09_cat,569,0.000956
10,ps_car_11,5,0.000008
11,ps_car_12,1,0.000002


In [22]:
# Usa ou moda ou média para preencher os valores "vazios" que nosso dataset contem, baseado nos metadados do mesmo
def fillNullNumbers(dfInput, dfMetadata, dfMissing, missing_default, label):

    from sklearn.impute import SimpleImputer

    media_imp = SimpleImputer(missing_values=missing_default, strategy='mean')
    moda_imp = SimpleImputer(missing_values=missing_default, strategy='most_frequent')

    for index,row in dfMissing.iterrows():
        columnName = row['column_name']
        columnType = dfMetadata.level[(dfMetadata.index == columnName)][0]

        if (columnType == 'interval'):
            imputerToUse = media_imp
            imputerString = 'media_imp'
        elif (columnType == 'ordinal'):
            imputerToUse = moda_imp
            imputerString = 'moda_imp'
        else:
            imputerToUse = None
            imputerString = None

        if (imputerToUse != None):
            dfInput[columnName] = imputerToUse.fit_transform(dfInput[[columnName]]).ravel()
            print(f"{label} - Preenchida coluna {columnName}, cujo tipo é {columnType}, usando o Imputer {imputerString}")

    return dfInput

In [23]:
dfTreino = fillNullNumbers(dfTreino, meta_train, missing_Train, -1, 'Treino')
print('---')
dfTeste = fillNullNumbers(dfTeste, meta_test, missing_Train, -1, 'Teste')

Treino - Preenchida coluna ps_reg_03, cujo tipo é interval, usando o Imputer media_imp
Treino - Preenchida coluna ps_car_11, cujo tipo é ordinal, usando o Imputer moda_imp
Treino - Preenchida coluna ps_car_12, cujo tipo é interval, usando o Imputer media_imp
Treino - Preenchida coluna ps_car_14, cujo tipo é interval, usando o Imputer media_imp
---
Teste - Preenchida coluna ps_reg_03, cujo tipo é interval, usando o Imputer media_imp
Teste - Preenchida coluna ps_car_11, cujo tipo é ordinal, usando o Imputer moda_imp
Teste - Preenchida coluna ps_car_12, cujo tipo é interval, usando o Imputer media_imp
Teste - Preenchida coluna ps_car_14, cujo tipo é interval, usando o Imputer media_imp


Os atributos categóricos podem ser mantidos porque o número de valores faltantes não é expressivo. Inclusive, a estratégia de preenchimento dos **atributos categóricos** é sempre mais complexa. Esses atributos **não se beneficiam de medidas estatísticas** como moda e média, portanto essas medidas não servem para preenchê-los de forma sintética.

---

## One-hot encoding (ou dummy variables)

Depois de ter tratado os dados faltantes, é importante que os dados ordinais tenham representação apropriada para o problema tratado. Se o dado não tem distância ou rankamento entre eles, cada valor de um atributo deve ser representado por um conjunto de atributos de mesma distância. *(Verificar slides desse encontro para que isso fique mais claro)*

Os dados que precisam ser separados em mais dimensões já foram identificados como nominais no pré-processamento. É importante verificar se esses dados têm grande variedade de valores ou não, e aplicar essa separação apenas se for viável. Por exemplo, se um determinado atributo tem 300 valores, isso geraria 300 colunas novas. Isso só se justificaria se fosse uma base realmente grande e se houvesse uma correlação muito alta entre essa variedade de valores e a classe.

In [24]:
def performOneHotEncoding(dfTrain, dfTest, meta_generic, dist_limit):
    v = meta_generic[(meta_generic.level == 'nominal') & (meta_generic.keep)].index
    display(v)
    for f in v:
        dist_values = dfTrain[f].value_counts().shape[0]
        print('Atributo {} tem {} valores distintos'.format(f, dist_values))
        if (dist_values > dist_limit):
            print('Atributo {} tem mais de {} valores distintos e por isso será ignorado'.format(f, dist_limit))
            dfTrain.drop([f], axis=1)
            v = v.drop([f])
        
    print('Antes do one-hot encoding tinha-se {} atributos'.format(dfTrain.shape[1]))
    dfTrain = pd.get_dummies(dfTrain, columns=v, drop_first=True)
    print('Depois do one-hot encoding tem-se {} atributos'.format(dfTrain.shape[1]))

    dfTest = pd.get_dummies(dfTest, columns=v, drop_first=True)
    missing_cols = set( dfTrain.columns ) - set( dfTest.columns )
    for c in missing_cols:
        dfTest[c] = 0

    dfTrain, dfTest = dfTrain.align(dfTest, axis=1)
    
    return dfTrain, dfTest

Vamos optar por manter todos atributos e, portanto, gerar o conjunto de atributos que os mantêm à mesma distância:

In [25]:
dfTreino, dfTeste = performOneHotEncoding(dfTreino, dfTeste, meta_train, 200)

Index(['ps_ind_02_cat', 'ps_ind_04_cat', 'ps_ind_05_cat', 'ps_car_01_cat',
       'ps_car_02_cat', 'ps_car_04_cat', 'ps_car_06_cat', 'ps_car_07_cat',
       'ps_car_08_cat', 'ps_car_09_cat', 'ps_car_10_cat', 'ps_car_11_cat'],
      dtype='object', name='varname')

Atributo ps_ind_02_cat tem 5 valores distintos
Atributo ps_ind_04_cat tem 3 valores distintos
Atributo ps_ind_05_cat tem 8 valores distintos
Atributo ps_car_01_cat tem 13 valores distintos
Atributo ps_car_02_cat tem 3 valores distintos
Atributo ps_car_04_cat tem 10 valores distintos
Atributo ps_car_06_cat tem 18 valores distintos
Atributo ps_car_07_cat tem 3 valores distintos
Atributo ps_car_08_cat tem 2 valores distintos
Atributo ps_car_09_cat tem 6 valores distintos
Atributo ps_car_10_cat tem 3 valores distintos
Atributo ps_car_11_cat tem 104 valores distintos
Antes do one-hot encoding tinha-se 56 atributos
Depois do one-hot encoding tem-se 210 atributos


In [26]:
from sklearn.preprocessing import MinMaxScaler

min_max_scaler = MinMaxScaler()

dfTreino[dfTreino.columns] = min_max_scaler.fit_transform(dfTreino[dfTreino.columns])
dfTeste[dfTeste.columns] = min_max_scaler.fit_transform(dfTeste[dfTeste.columns])

In [27]:
dfTeste.drop(['target'], axis=1, inplace=True)

## Depois de todo pré-processamento...

É hora de verificar se tanto treino como teste têm o mesmo tamanho/formato, e aplicar um modelo de classificação já que esse é um problema desse tipo. Vale lembrar que o tamanho do treino e teste pode variar quando você estiver participando de outras competições ou explorando outros conjuntos de dados.

Isso porque na maioria das competições não se tem o *target* do test. Estima-se uma resposta e submete ao Kaggle, por exemplo, para que ele verifique qual foi o resultado final. Então esse tamanho pode variar em 1 entre treino e teste. No nosso caso, como todos os dados vêm de uma mesma fonte para experimentos, é esperado que tenham a mesma quantidade de atributos ou colunas.

In [28]:
# Models
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

# Feature Selection
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV, cross_val_score, ShuffleSplit

# Auxiliary Scores
from sklearn.metrics import recall_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import log_loss
from sklearn.metrics import confusion_matrix
from sklearn.metrics import precision_score
from sklearn.metrics import f1_score

In [29]:
def showDistribution(val_classes):
    nonUsed, used = pd.DataFrame(val_classes).groupby('target').size()
    print('---')
    print(f'Das {pd.DataFrame(val_classes).shape[0]} entradas no dataset, {nonUsed} foram de casos onde não foi acionado o seguro e {used} foram caso onde houve acionamento')
    print(f'Temos assim {round((used/len(val_classes)) * 100,6)}% de ocorrencias em que o resultado (1 ou "houve acionamento") desejamos prever')
    print('---')

In [30]:
def logisticRegression(X_Train, y_Train, X_Val, y_Val):

    model = LogisticRegression(solver='lbfgs')

    model.fit(X_Train, y_Train)

    y_pred_class = model.predict(X_Val)
    y_pred_proba = model.predict_proba(X_Val)

    recall = recall_score(y_Val, y_pred_class)
    accuracy = accuracy_score(y_Val, y_pred_class)
    logloss = log_loss(y_Val, y_pred_proba)
    precision =  precision_score(y_Val, y_pred_class)
    f1 = f1_score(y_Val, y_pred_class)
    gini = eval_gini(y_Val, y_pred_class)

    print(f'Baseline - Regressão Logistica')
    print('---')
    print(f'Acurácia: {round(accuracy, 6)}%')
    print(f'Recall: {round(recall, 6)}%')
    print(f'Precisão: {round(precision, 6)}%')
    print(f'Log Loss: {round(logloss, 6)}')
    print(f'F1 Score: {round(f1, 6)}')
    print(f'Gini: {round(gini, 6)}')

    print('---')
    print('Matriz de Confusão')
    display(pd.DataFrame(confusion_matrix(y_Val, y_pred_class)))
    print('---')
    
    return model, 'Baseline - Regressão Logistica'

In [31]:
# n_estimators=20, learning_rate = 0.5, max_features=2, max_depth = 70, random_state = 0 - Recall: 0.057621%

def xGBClassifier(X_Train, y_Train, X_Val, y_Val, modelName, modelParams):

    if (modelParams == None):
        clf = XGBClassifier()
    else:
        clf = XGBClassifier(**modelParams)  
        modelName = modelName + ' - Parameters: ' + str(modelParams)
    
    clf.fit(X_Train, y_Train)

    y_pred_class = clf.predict(X_Val)
    y_pred_proba = clf.predict_proba(X_Val)

    recall = recall_score(y_Val, y_pred_class)
    accuracy = accuracy_score(y_Val, y_pred_class)
    logloss = log_loss(y_Val, y_pred_proba)
    gini = eval_gini(y_Val, y_pred_class)
    precision =  precision_score(y_Val, y_pred_class)
    f1 = f1_score(y_Val, y_pred_class)

    print(modelName)
    print('---')
    print(f'Acurácia: {round(accuracy, 6)}%')
    print(f'Recall: {round(recall, 6)}%')
    print(f'Precisão: {round(precision, 6)}%')
    print(f'Log Loss: {round(logloss, 6)}')
    print(f'F1 Score: {round(f1, 6)}')
    print(f'Gini: {round(gini, 6)}')

    print('---')
    print('Matriz de Confusão')
    display(pd.DataFrame(confusion_matrix(y_Val, y_pred_class)))
    print('---')
    
    return clf, modelName

In [32]:
def decisionTreeClassifier(X_Train, y_Train, X_Val, y_Val):

    clf = DecisionTreeClassifier()

    clf.fit(X_Train, y_Train)

    y_pred_class = clf.predict(X_Val)
    y_pred_proba = clf.predict_proba(X_Val)

    recall = recall_score(y_Val, y_pred_class)
    accuracy = accuracy_score(y_Val, y_pred_class)
    gini = eval_gini(y_Val, y_pred_class)
    logloss = log_loss(y_Val, y_pred_proba)
    precision =  precision_score(y_Val, y_pred_class)
    f1 = f1_score(y_Val, y_pred_class)

    print(f'Decision Tree - Default Parameters')
    print('---')
    print(f'Acurácia: {round(accuracy, 6)}%')
    print(f'Recall: {round(recall, 6)}%')
    print(f'Precisão: {round(precision, 6)}%')
    print(f'Log Loss: {round(logloss, 6)}')
    print(f'F1 Score: {round(f1, 6)}')
    print(f'Gini: {round(gini, 6)}')

    print('---')
    print('Matriz de Confusão')
    display(pd.DataFrame(confusion_matrix(y_Val, y_pred_class)))
    print('---')
    
    return clf, f'Decision Tree - Default Parameters'

In [33]:
def gridSearchKNN(X_Train, y_Train, X_Val, y_Val, k_range):
    clf=KNeighborsClassifier()
    param_grid=dict(n_neighbors=k_range)
    scores = ['f1']
    for sc in scores:
        grid=GridSearchCV(clf,param_grid,cv=4,scoring=sc,n_jobs=-1)
        print("K-Nearest Neighbors - Tuning hyper-parameters for %s" % sc)
        
        grid.fit(X_Train,y_Train)
        
        print(grid.best_params_)
        print(np.round(grid.best_score_,3))
        
        y_pred_class = grid.predict(X_Val)
        y_pred_proba = grid.predict_proba(X_Val)

        recall = recall_score(y_Val, y_pred_class)
        accuracy = accuracy_score(y_Val, y_pred_class)
        gini = eval_gini(y_Val, y_pred_class)
        logloss = log_loss(y_Val, y_pred_proba)
        precision =  precision_score(y_Val, y_pred_class)
        f1 = f1_score(y_Val, y_pred_class)

        print(f'KNN with recall-maxing hyperparameters - {grid.best_params_}')
        print('---')
        print(f'Acurácia: {round(accuracy, 6)}%')
        print(f'Recall: {round(recall, 6)}%')
        print(f'Precisão: {round(precision, 6)}%')
        print(f'Log Loss: {round(logloss, 6)}')
        print(f'F1 Score: {round(f1, 6)}')
        print(f'Gini: {round(gini, 6)}')

        print('---')
        print('Matriz de Confusão')
        display(pd.DataFrame(confusion_matrix(y_Val, y_pred_class)))
        print('---')
        
        return grid, f'KNN with recall-maxing hyperparameters - {grid.best_params_}'

In [34]:
def gridSearchSVC(X_Train, y_Train, X_Val, y_Val):
    svc=SVC()
    param_grid = [{'kernel': ['rbf'], 'gamma': [1e-3, 1e-4, 1e-5],'C': [1, 10, 100, 1000]},
                  {'kernel': ['linear'], 'C': [1, 10, 100, 1000]}]
    scores = ['f1']
    for sc in scores:
        grid=GridSearchCV(svc,param_grid,cv=4,scoring=sc,n_jobs=-1)
        
        print("Support Vector Classifier - Tuning hyper-parameters for %s" % sc)
        
        grid.fit(X_Train,y_Train)
        print(grid.best_params_)
        print(np.round(grid.best_score_,3))
        
        y_pred_class = grid.predict(X_Val)

        recall = recall_score(y_Val, y_pred_class)
        accuracy = accuracy_score(y_Val, y_pred_class)
        gini = eval_gini(y_Val, y_pred_class)
        precision =  precision_score(y_Val, y_pred_class)
        f1 = f1_score(y_Val, y_pred_class)

        print(f'SVC with recall-maxing hyperparameters - {grid.best_params_}')
        print('---')
        print(f'Acurácia: {round(accuracy, 6)}%')
        print(f'Recall: {round(recall, 6)}%')
        print(f'Precisão: {round(precision, 6)}%')
        print(f'F1 Score: {round(f1, 6)}')
        print(f'Gini: {round(gini, 6)}')

        print('---')
        print('Matriz de Confusão')
        display(pd.DataFrame(confusion_matrix(y_Val, y_pred_class)))
        print('---')
        
        return grid, f'SVC with recall-maxing hyperparameters - {grid.best_params_}'

In [35]:
def gridSearchXGB(X_Train, y_Train, X_Val, y_Val, score):
    xgb=XGBClassifier(random_state = 0)
    param_grid = [{'n_estimators': [100, 200, 300, 400], 'learning_rate': [0.1, 0.25, 0.5, 0.75],'max_depth': [25, 50, 75, 100], 'gamma': [0, 3, 6, 9]}]
    scores = [score]
    for sc in scores:
        grid=GridSearchCV(xgb,param_grid,cv=2,scoring=sc,n_jobs=-1)
        
        print("XGBoost - Tuning hyper-parameters for %s" % sc)
        
        grid.fit(X_Train,y_Train)
        print(grid.best_params_)
        print(np.round(grid.best_score_,3))
        
        y_pred_class = grid.predict(X_Val)

        recall = recall_score(y_Val, y_pred_class)
        accuracy = accuracy_score(y_Val, y_pred_class)
        gini = eval_gini(y_Val, y_pred_class)
        precision =  precision_score(y_Val, y_pred_class)
        f1 = f1_score(y_Val, y_pred_class)

        print(f'XGBoost with {sc}-maxing hyperparameters - {grid.best_params_}')
        print('---')
        print(f'Acurácia: {round(accuracy, 6)}%')
        print(f'Recall: {round(recall, 6)}%')
        print(f'Precisão: {round(precision, 6)}%')
        print(f'F1 Score: {round(f1, 6)}')
        print(f'Gini: {round(gini, 6)}')

        print('---')
        print('Matriz de Confusão')
        display(pd.DataFrame(confusion_matrix(y_Val, y_pred_class)))
        print('---')
        
        return grid, f'XGBoost with {sc}-maxing hyperparameters - {grid.best_params_}'

In [36]:
def predictTestDataset(X_Test, y_Test, clfModel, clfName):
    y_pred_class = clfModel.predict(X_Test)
    y_pred_proba = clfModel.predict_proba(X_Test)

    recall = recall_score(y_Test, y_pred_class)
    accuracy = accuracy_score(y_Test, y_pred_class)
    gini = eval_gini(y_Test, y_pred_class)
    logloss = log_loss(y_Test, y_pred_proba)
    precision =  precision_score(y_Test, y_pred_class)
    f1 = f1_score(y_Test, y_pred_class)

    print(clfName)
    print('---')
    print(f'Acurácia: {round(accuracy, 6)}%')
    print(f'Recall: {round(recall, 6)}%')
    print(f'Precisão: {round(precision, 6)}%')
    print(f'Log Loss: {round(logloss, 6)}')
    print(f'F1 Score: {round(f1, 6)}')
    print(f'Gini: {round(gini, 6)}')

    print('---')
    print('Matriz de Confusão')
    display(pd.DataFrame(confusion_matrix(y_Test, y_pred_class)))
    print('---')

In [37]:
def predictContestDataset(X_Test, clfModel, clfName):
    
    print(clfName)
    print('---')
    
    y_pred_class = clfModel.predict(X_Test)
    y_pred_proba = clfModel.predict_proba(X_Test)
    
    pd_prediction = pd.DataFrame(y_pred_class)
    pd_prediction.columns = ['target']
    showDistribution(pd_prediction)

    return y_pred_class, y_pred_proba

In [38]:
print(dfTreino.shape)
print(dfTeste.shape)

(595212, 210)
(892816, 209)


In [39]:
sample_size = 10000
inactive_sample_size = int(sample_size * 1)

activated_indices = dfTreino[dfTreino.target == 1].index
activated = dfTreino.loc[np.random.choice(activated_indices, sample_size, replace=False)]

inactive_indices = dfTreino[dfTreino.target == 0].index
inactive = dfTreino.loc[np.random.choice(inactive_indices, inactive_sample_size, replace=False)]

subsampled = pd.concat([activated, inactive])

subsampled.sort_index(inplace=True)

In [40]:
X = subsampled.drop(['target'], axis=1)
y = subsampled['target']

In [41]:
X_supersampled = dfTreino.drop(X.index).drop(['target'], axis=1)
y_supersampled = dfTreino.drop(X.index)['target']

In [42]:
print(X_supersampled.shape)
print(y_supersampled.shape)

(575212, 209)
(575212,)


In [43]:
print(X.shape)
print(y.shape)

(20000, 209)
(20000,)


In [44]:
showDistribution(y)

---
Das 20000 entradas no dataset, 10000 foram de casos onde não foi acionado o seguro e 10000 foram caso onde houve acionamento
Temos assim 50.0% de ocorrencias em que o resultado (1 ou "houve acionamento") desejamos prever
---


In [45]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)

showDistribution(y_train)
logRegModel, logRegName = logisticRegression(X_train, y_train, X_val, y_val)
xgbPureModel, xgbPureName = xGBClassifier(X_train, y_train, X_val, y_val, 'XGBoost - Base',None)
#xgbPresetModel, xgbPresetName = xGBClassifier(X_train, y_train, X_val, y_val, 'XGBoost - Preset', {'n_estimator':400, 'learning_rate' : 0.5,'random_state' : 0,'max_depth':70,'objective':"binary:logistic",'subsample':.8,'min_child_weig':6,'colsample_bytr':.8,'scale_pos_weight':1.6, 'gamma':10, 'reg_alph':8, 'reg_lambda':1})
xgbHyperParametrizedModel, xgbHyperParametrizedName = xGBClassifier(X_train, y_train, X_val, y_val, 'XGBoost - Hyperparametrized',{'colsample_bytree': 0.8, 'learning_rate': 0.2, 'max_depth': 2, 'min_child_weight': 16, 'n_estimators': 100, 'subsample': 1.0})
#decTreeModel, decTreeName = decisionTreeClassifier(X_train, y_train, X_val, y_val)
#knnModel, knnName = gridSearchKNN(X_train, y_train, X_val, y_val, list(range(1,20)))
#svcModel, svcName = gridSearchSVC(X_train, y_train, X_val, y_val)

showDistribution(y_supersampled)
predictTestDataset(X_supersampled, y_supersampled, logRegModel, logRegName)
predictTestDataset(X_supersampled, y_supersampled, xgbPureModel, xgbPureName)
#predictTestDataset(X_supersampled, y_supersampled, xgbPresetModel, xgbPresetName)
predictTestDataset(X_supersampled, y_supersampled, xgbHyperParametrizedModel, xgbHyperParametrizedName)
#predictTestDataset(X_supersampled, y_supersampled, decTreeModel, decTreeName)
#predictTestDataset(X_supersampled, y_supersampled, knnModel, knnName)
#predictTestDataset(X_supersampled, y_supersampled, svcModel, svcName)

---
Das 14000 entradas no dataset, 7000 foram de casos onde não foi acionado o seguro e 7000 foram caso onde houve acionamento
Temos assim 50.0% de ocorrencias em que o resultado (1 ou "houve acionamento") desejamos prever
---
Baseline - Regressão Logistica
---
Acurácia: 0.581833%
Recall: 0.544333%
Precisão: 0.588468%
Log Loss: 0.670823
F1 Score: 0.565541
Gini: 0.138942
---
Matriz de Confusão


,0,1
0,1858,1142
1,1367,1633


---
XGBoost - Base
---
Acurácia: 0.592167%
Recall: 0.559%
Precisão: 0.598715%
Log Loss: 0.665354
F1 Score: 0.578176
Gini: 0.17279
---
Matriz de Confusão


,0,1
0,1876,1124
1,1323,1677


---
XGBoost - Hyperparametrized - Parameters: {'colsample_bytree': 0.8, 'learning_rate': 0.2, 'max_depth': 2, 'min_child_weight': 16, 'n_estimators': 100, 'subsample': 1.0}
---
Acurácia: 0.595833%
Recall: 0.562667%
Precisão: 0.602642%
Log Loss: 0.66531
F1 Score: 0.581969
Gini: 0.173478
---
Matriz de Confusão


...

---
XGBoost - Base
---
Acurácia: 0.622075%
Recall: 0.559176%
Precisão: 0.02989%
Log Loss: 0.667164
F1 Score: 0.056746
Gini: 0.185517
---
Matriz de Confusão


,0,1
0,351286,212232
1,5155,6539


---
XGBoost - Hyperparametrized - Parameters: {'colsample_bytree': 0.8, 'learning_rate': 0.2, 'max_depth': 2, 'min_child_weight': 16, 'n_estimators': 100, 'subsample': 1.0}
---
Acurácia: 0.620809%
Recall: 0.559347%
Precisão: 0.029799%
Log Loss: 0.667336
F1 Score: 0.056584
Gini: 0.181594
---
Matriz de Confusão


,0,1
0,350556,212962
1,5153,6541


---


In [46]:
#showDistribution(y_train)
#xgbGridSearchModel, xgbGridSearchName = gridSearchXGB(X_train, y_train, X_val, y_val, 'f1')

#showDistribution(y_supersampled)
#predictTestDataset(X_supersampled, y_supersampled, xgbGridSearchModel, xgbGridSearchName)

In [47]:
contest_prediction, contest_prediction_probability = predictContestDataset(dfTeste, xgbHyperParametrizedModel, xgbHyperParametrizedName)

XGBoost - Hyperparametrized - Parameters: {'colsample_bytree': 0.8, 'learning_rate': 0.2, 'max_depth': 2, 'min_child_weight': 16, 'n_estimators': 100, 'subsample': 1.0}
---
---
Das 892816 entradas no dataset, 609077 foram de casos onde não foi acionado o seguro e 283739 foram caso onde houve acionamento
Temos assim 31.780232% de ocorrencias em que o resultado (1 ou "houve acionamento") desejamos prever
---


In [48]:
sample    = pd.read_csv('../input/sample_submission.csv', low_memory=False)
sample.target = contest_prediction_probability
sample.target = 1 - sample.target
sample.to_csv("submission.csv", float_format='%.6f', index=False)